In [11]:
%%javascript
(() => {
  // 只隐藏编辑器，不隐藏 cell 的 toolbar / prompt
  const selectors = ['.jp-InputArea-editor', '.cm-editor', '.CodeMirror'];

  // 找到当前 Notebook 使用的编辑器 DOM（优先匹配第一个存在的 selector）
  function getEditors() {
    for (const s of selectors) {
      const nodes = document.querySelectorAll(s);
      if (nodes.length) return { sel: s, nodes };
    }
    return { sel: null, nodes: [] };
  }

  // 切换显示/隐藏
  function toggle() {
    const { sel, nodes } = getEditors();
    if (!nodes.length) return alert('没找到编辑器区域：' + selectors.join(' / '));

    const hide = nodes[0].style.display !== 'none';
    nodes.forEach(n => n.style.display = hide ? 'none' : '');

    // 仅用于调试：输出当前使用的 selector
    console.log("toggle selector:", sel);
  }

  // 创建右上角按钮（避免重复创建）
  let btn = document.getElementById('toggleCodeBtn');
  if (!btn) {
    btn = document.createElement('button');
    btn.id = 'toggleCodeBtn';
    btn.textContent = 'Hide/Show Code';
    btn.style.cssText =
      'position:fixed;top:12px;right:12px;z-index:99999;padding:6px 12px;border-radius:6px;';
    btn.addEventListener('click', toggle);
    document.body.appendChild(btn);
  }

  // 默认隐藏编辑器（只隐藏代码，不影响按钮/工具栏/输出）
  const { nodes } = getEditors();
  nodes.forEach(n => n.style.display = 'none');
})();

<IPython.core.display.Javascript object>

# 市场趋势判断方法验证

> **模块定位**: 建立和验证市场环境判断方法，为01_market_trend_comprehensive提供方法论支撑
>
> **目标**: 通过10年历史数据回测，验证各种市场判断指标的有效性，优化参数阈值

---

## 📋 目录

1. [方法论框架](#1-方法论框架)
2. [数据源与可用性](#2-数据源与可用性)
3. [市场状态量化定义](#3-市场状态量化定义)
4. [Phase 1: 快速验证回测](#4-phase-1-快速验证回测)
5. [Phase 2: 完整10年回测](#5-phase-2-完整10年回测)
6. [结果分析与可视化](#6-结果分析与可视化)
7. [参数优化建议](#7-参数优化建议)
8. [结论与后续改进](#8-结论与后续改进)

In [1]:
# 环境初始化
import sys
from pathlib import Path

# 添加项目根目录到 Python 路径
project_root = Path.cwd()
while project_root.name != 'TRQuant' and project_root.parent != project_root:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"项目根目录: {project_root}")

# 基础库
import logging
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from IPython.display import display, Markdown, HTML

# 配置日志
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# 可视化
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

print("✅ 环境初始化完成")

项目根目录: /home/taotao/.cursor/worktrees/TRQuant
✅ 环境初始化完成


---

## 1. 方法论框架

### 1.1 核心理念

市场趋势判断基于**多周期共振**原理：

```
长期趋势 (权重50%) + 中期趋势 (权重30%) + 短期趋势 (权重20%) = 综合判断
```

### 1.2 三周期定义

| 周期 | 时间窗口 | 分析天数 | 权重 | 核心指标 | 验证期 |
|------|----------|----------|------|----------|--------|
| 短期 | 1-8周 | 5-40日 | 20% | MA5/MA10/RSI14/KDJ | 5日 |
| 中期 | 9-24周 | 45-120日 | 30% | MA20/MA60/MACD/布林带 | 20日 |
| 长期 | 25-48周 | 125-240日 | 50% | MA120/MA250/ADX/月线趋势 | 60日 |

### 1.3 A股特色指标

| 指标 | 数据源 | 权重 | 说明 |
|------|--------|------|------|
| 北向资金 | JQData | 35% | 外资态度，2014.11-2024.08有完整买卖数据， AKshare可以补充其余时间段数据 |
| 融资融券 | JQData | 25% | 市场杠杆水平 |
| 市场宽度 | JQData | 40% | 涨跌停比、涨跌家数比 |

### 1.4 量化计算公式详解

#### 1.4.1 8维技术指标得分计算

每个指标得分范围 **-100 ~ +100**，综合后加权平均：

**1. 均线系统得分 (MA_score, 权重20%)**:

```
MA排列得分 = {
    MA5 > MA10 > MA20 > MA60: +40 (多头排列)
    MA5 > MA10 > MA20:        +20
    MA60 < 价格 < MA5:        +10
    MA60 > MA20 > MA10 > MA5: -40 (空头排列)
    其他:                      0
}

价格位置得分 = {
    价格 > MA5 * 1.05: +30 (强势)
    价格 > MA20:       +10
    价格 < MA60:       -20
    价格 < MA120:      -30
}

MA_score = MA排列得分 + 价格位置得分
```

**📝 数值示例**:
```
假设当前: 收盘价=3200, MA5=3180, MA10=3150, MA20=3100, MA60=3050, MA120=3000
→ MA5 > MA10 > MA20 > MA60 → 多头排列 +40分
→ 价格3200 > MA5*1.05=3339? 否; 价格 > MA20=3100? 是 → +10分
→ MA_score = 40 + 10 = +50
```

**2. MACD得分 (MACD_score, 权重18%)**:

```
MACD_score = DIF位置得分 + 柱状图得分 + 金叉死叉得分

DIF位置 = {
    DIF > 0 且 DIF > DEA: +30
    DIF > 0 且 DIF < DEA: +10
    DIF < 0 且 DIF > DEA: -10
    DIF < 0 且 DIF < DEA: -30
}

柱状图 = {
    MACD柱 > 0 且 连续3日放大: +20
    MACD柱 > 0:               +10
    MACD柱 < 0 且 连续3日放大: -20
    MACD柱 < 0:               -10
}

金叉死叉 = {
    5日内DIF上穿DEA: +20
    5日内DIF下穿DEA: -20
}
```

**📝 数值示例**:
```
假设: DIF=15, DEA=12, MACD柱=3(红柱放大第2天)
→ DIF > 0 且 DIF > DEA → +30分
→ MACD柱 > 0 → +10分
→ 无金叉死叉 → 0分
→ MACD_score = 30 + 10 + 0 = +40
```

**3. RSI得分 (RSI_score, 权重12%)**:

```
RSI_score = {
    RSI14 > 80:           -40 (极度超买)
    RSI14 > 70:           -20 (超买)
    RSI14 > 50:           +20 (偏强)
    RSI14 > 30:           -10 (偏弱)
    RSI14 > 20:           +10 (超卖反弹机会)
    RSI14 < 20:           +30 (极度超卖)
} + RSI背离调整

RSI背离 = {
    价格新高但RSI未新高: -30 (顶背离)
    价格新低但RSI未新低: +30 (底背离)
}
```

**4. 布林带得分 (BB_score, 权重12%)**:

```
BB_score = {
    价格 > 上轨:          -20 (超买警告)
    价格 > 中轨且带宽扩张: +20 (突破)
    价格 > 中轨:          +10
    价格 < 中轨:          -10
    价格 < 下轨:          +15 (超卖)
    带宽 < 5%:            ±0 (收窄待突破)
}
```

**5. 成交量得分 (VOL_score, 权重12%)**:

```
VOL_score = {
    量价齐升(价涨量增50%+): +30
    价涨量增:              +15
    价涨量缩:              +5 (缩量上涨)
    价跌量增:              -20 (恐慌出逃)
    价跌量缩:              -5 (惜售)
    地量(低于20日均量50%): +10 (可能见底)
}
```

**6-8. KDJ/ADX/MFI得分**: 类似逻辑，根据超买超卖区间和趋势方向计算。

#### 1.4.2 三周期综合得分公式

```
周期得分 = Σ(指标原始得分 × 指标权重)

短期得分(S) = 40日数据计算的加权得分
中期得分(M) = 120日数据计算的加权得分  
长期得分(L) = 240日数据计算的加权得分

综合得分 = S × 0.20 + M × 0.30 + L × 0.50
```

**📝 完整计算示例** (2024年1月15日，上证指数):

```
原始数据:
- 收盘价: 2887点
- MA5=2895, MA10=2920, MA20=2950, MA60=3020, MA120=3080, MA250=3150
- MACD: DIF=-18, DEA=-12, 柱状图=-6(绿柱缩小)
- RSI14=35
- 成交量: 较20日均量+15%

短期(40日)各指标得分:
| 指标 | 权重 | 得分 | 说明 |
|------|------|------|------|
| MA   | 20%  | -50  | 空头排列(-40)+价格<MA60(-10) |
| MACD | 18%  | -25  | DIF<0且<DEA(-30)+绿柱缩小(+5) |
| RSI  | 12%  | +5   | RSI14=35在30-50区间 |
| BB   | 12%  | -15  | 价格在中轨下方 |
| VOL  | 12%  | -5   | 价跌量增 |
| KDJ  | 10%  | +10  | K=28,D=32,超卖区 |
| ADX  | 8%   | +15  | ADX=32,趋势明确 |
| MFI  | 8%   | -10  | 资金流出 |

短期得分 = -50×0.20 + (-25)×0.18 + 5×0.12 + (-15)×0.12 
         + (-5)×0.12 + 10×0.10 + 15×0.08 + (-10)×0.08
         = -10 - 4.5 + 0.6 - 1.8 - 0.6 + 1.0 + 1.2 - 0.8
         = -14.9 (短期看空)

中期得分(类似计算) = -28.5 (中期看空)
长期得分(类似计算) = -35.2 (长期看空)

综合得分 = (-14.9) × 0.20 + (-28.5) × 0.30 + (-35.2) × 0.50
         = -2.98 - 8.55 - 17.6
         = -29.1

→ 判定: 熊市反弹 (长期<-30附近, 中期<-20)
→ 建议仓位: 10-30%
```

#### 1.4.3 A股特色指标计算公式

**1. 北向资金得分 (North_score)**

```
日净买入得分 = {
    净买入 > 80亿:  +30 (大幅流入)
    净买入 > 30亿:  +15 (流入)
    净买入 > 0:     +5
    净流出 < -30亿: -15 (流出)
    净流出 < -80亿: -30 (大幅流出)
}

5日累计得分 = {
    累计 > 150亿:   +40 (持续大幅流入)
    累计 > 80亿:    +25
    累计 > 50亿:    +15
    累计 < -50亿:   -15
    累计 < -100亿:  -30
}

North_score = 日净买入得分 + 5日累计得分
```

**📝 数值示例**:
```
假设: 今日净买入+52亿, 5日累计+180亿
→ 日净买入得分: 52亿 > 30亿 → +15分
→ 5日累计得分: 180亿 > 150亿 → +40分
→ North_score = 15 + 40 = +55
```

**2. 融资融券得分 (Margin_score)**

```
融资变化率得分 = {
    日变化率 > 2%:   +30 (杠杆增加)
    日变化率 > 1%:   +15
    日变化率 > 0.5%: +5
    日变化率 < -0.5%: -10
    日变化率 < -1%:  -20
    日变化率 < -2%:  -35 (去杠杆)
}

融资余额水平 = {
    余额 > 历史80%分位: -10 (杠杆过高风险)
    余额 < 历史20%分位: +10 (杠杆低位机会)
}

Margin_score = 融资变化率得分 + 融资余额水平
```

**3. 市场宽度得分 (Breadth_score)**

```
涨跌停比得分 = {
    涨停数/跌停数 > 5:  +40 (极度强势)
    涨停数/跌停数 > 3:  +25
    涨停数/跌停数 > 2:  +15
    涨停数/跌停数 < 0.5: -20
    涨停数/跌停数 < 0.3: -35 (极度弱势)
}

涨跌家数比 = {
    上涨/下跌 > 2:     +20
    上涨/下跌 > 1.5:   +10
    上涨/下跌 < 0.7:   -10
    上涨/下跌 < 0.5:   -20
}

均线多头占比 = {
    站上MA20占比 > 60%: +15
    站上MA20占比 > 40%: +5
    站上MA20占比 < 30%: -10
    站上MA20占比 < 20%: -20
}

Breadth_score = 涨跌停比得分 + 涨跌家数比 + 均线多头占比
```

**📝 完整A股指标综合示例**:
```
假设当日数据:
- 北向资金: 日净买入+25亿, 5日累计+60亿
- 融资余额: 1.65万亿, 日变化+0.3%
- 涨停52家, 跌停8家, 上涨2100家, 下跌2600家
- 站上MA20: 38%

North_score = 5(日净买入0~30亿) + 15(5日累计>50亿) = +20
Margin_score = 5(变化率0.3%>0) + 0(余额中性) = +5
Breadth_score = 25(涨跌停比6.5>3) + (-10)(涨跌比0.8<1) + 5(MA占比38%>30%) = +20

A股特色综合 = 20 × 0.35 + 5 × 0.25 + 20 × 0.40
            = 7 + 1.25 + 8 = +16.25

→ 信号: 中性偏多
→ 特征: 北向资金温和流入，涨停家数多但整体涨跌家数偏弱
```

#### 1.4.4 市场状态判断算法

**状态判断流程**:

```python
def determine_market_state(L, M, S):
    """
    L: 长期得分 (-100 ~ +100)
    M: 中期得分 (-100 ~ +100)
    S: 短期得分 (-100 ~ +100)
    """
    # 牛市系列 (长期>30)
    if L > 30:
        if M > 20 and S > 10:
            return "牛市确认(共振)", "80-100%"
        elif M > 20:
            return "牛市确认", "70-90%"
        elif M > 0:
            return "牛市震荡", "50-70%"
        elif S < -20:
            return "牛市短期调整", "40-60%"
        else:
            return "牛市中期调整", "30-50%"
    
    # 熊市系列 (长期<-30)
    elif L < -30:
        if M < -20 and S < -10:
            return "熊市确认(共振)", "0-10%"
        elif M < -20:
            return "熊市确认", "0-20%"
        elif M < 0:
            return "熊市反弹", "10-30%"
        elif S > 20:
            return "熊市技术反弹", "20-40%"
        else:
            return "熊市筑底", "20-40%"
    
    # 震荡系列 (-30 <= 长期 <= 30)
    else:
        if M > 10 and S > 10:
            return "突破在即", "50-70%"
        elif M < -10 and S < -10:
            return "破位风险", "10-30%"
        elif S > 20 and M > 0:
            return "复苏初期", "40-60%"
        elif S < -20 and M < 0:
            return "见顶回落", "20-40%"
        else:
            return "窄幅震荡", "30-50%"
```

**📝 判断示例**:

| 日期 | 长期(L) | 中期(M) | 短期(S) | 状态 | 仓位 |
|------|---------|---------|---------|------|------|
| 2015-06-01 | +55 | +48 | +62 | 牛市确认(共振) | 80-100% |
| 2015-07-15 | +35 | -25 | -55 | 牛市中期调整 | 30-50% |
| 2018-10-15 | -45 | -38 | -28 | 熊市确认(共振) | 0-10% |
| 2019-01-05 | -35 | +5 | +25 | 熊市技术反弹 | 20-40% |
| 2020-03-25 | -8 | +15 | +32 | 复苏初期 | 40-60% |
| 2023-08-01 | +12 | -5 | -28 | 见顶回落 | 20-40% |

#### 1.4.5 验证准确率定义

**1. 信号方向准确率**:

```
对于看多信号(综合得分 > +30):
  正确条件 = 后续N日收益率 > 0

对于看空信号(综合得分 < -30):
  正确条件 = 后续N日收益率 < 0

对于中性信号(-30 <= 综合得分 <= +30):
  正确条件 = |后续N日收益率| < 2%

准确率 = 正确信号数 / 总信号数 × 100%
```

**2. 三周期分别验证**:

| 周期 | 验证期限 | 目标准确率 | 说明 |
|------|----------|------------|------|
| 短期 | 5个交易日 | >55% | 高频但噪音多 |
| 中期 | 20个交易日 | >60% | 主要决策依据 |
| 长期 | 60个交易日 | >65% | 最可靠信号 |

**3. 市场状态识别准确率**:

```
牛市状态(含共振/确认/震荡/调整):
  正确条件 = 后续60日最大收益 > 5%

熊市状态(含共振/确认/反弹/筑底):
  正确条件 = 后续60日最大回撤 > 5%

震荡状态:
  正确条件 = 后续60日振幅在±8%以内
```

**4. 分市场周期验证标准**:

| 市场周期 | 时间段 | 特征 | 重点验证 |
|----------|--------|------|----------|
| 2015牛熊 | 2015.01-2016.01 | 暴涨暴跌 | 反转信号捕捉 |
| 慢熊调整 | 2018.01-2019.01 | 持续阴跌 | 熊市识别准确性 |
| 疫情冲击 | 2020.01-2020.04 | V型反转 | 底部信号灵敏度 |
| 结构行情 | 2021.01-2023.12 | 分化严重 | 震荡期仓位控制 |

---

## 2. 数据源与可用性

### 2.1 JQData 数据范围

| 数据类型 | API | 起始日期 | 结束日期 | 说明 |
|----------|-----|----------|----------|------|
| 指数价格 | `get_price` | 2005年 | 至今 | 完全覆盖 |
| 北向资金(买卖分项) | `STK_ML_QUOTA` | 2014-11-17 | 2024-08-16 | 之后仅有成交总额 |
| 融资融券 | `STK_MT_TOTAL` | 2010年 | 至今 | 完全覆盖 |
| 涨跌停 | `get_price` + `high_limit/low_limit` | 2005年 | 至今 | 完全覆盖 |

### 2.2 回测时间范围选择

基于数据可用性，选择以下回测区间：

- **Phase 1 (快速验证)**: 2023-01-01 ~ 2024-08-16 (约1.5年)
- **Phase 2 (完整回测)**: 2014-11-17 ~ 2024-08-16 (约10年)

---

## 3. 市场状态量化定义

### 3.1 14种市场状态

基于短中长三周期得分，定义14种市场状态：

#### 🟢 牛市系列 (5种)

| 状态 | 长期得分 | 中期得分 | 短期得分 | 建议仓位 | 说明 |
|------|----------|----------|----------|----------|------|
| 牛市确认(共振) | >30 | >20 | >10 | 80-100% | 全周期共振看多 |
| 牛市确认 | >30 | >20 | 任意 | 70-90% | 长中期看多 |
| 牛市震荡 | >30 | 0~20 | 任意 | 50-70% | 牛市中的整理 |
| 牛市短期调整 | >30 | 任意 | <-20 | 40-60% | 短期回调，可逢低 |
| 牛市中期调整 | >30 | <0 | 任意 | 30-50% | 中期调整，谨慎 |

#### 🔴 熊市系列 (5种)

| 状态 | 长期得分 | 中期得分 | 短期得分 | 建议仓位 | 说明 |
|------|----------|----------|----------|----------|------|
| 熊市确认(共振) | <-30 | <-20 | <-10 | 0-10% | 全周期共振看空 |
| 熊市确认 | <-30 | <-20 | 任意 | 0-20% | 长中期看空 |
| 熊市反弹 | <-30 | -20~0 | 任意 | 10-30% | 反弹持续性存疑 |
| 熊市技术反弹 | <-30 | 任意 | >20 | 20-40% | 短线可参与 |
| 熊市筑底 | <-30 | >0 | 任意 | 20-40% | 可能出现转机 |

#### 🟡 震荡系列 (4种)

| 状态 | 长期得分 | 中期得分 | 短期得分 | 建议仓位 | 说明 |
|------|----------|----------|----------|----------|------|
| 突破在即 | -30~30 | >10 | >10 | 50-70% | 震荡上沿 |
| 破位风险 | -30~30 | <-10 | <-10 | 10-30% | 震荡下沿 |
| 复苏初期 | -30~30 | >0 | >20 | 40-60% | 短期走强 |
| 见顶回落 | -30~30 | <0 | <-20 | 20-40% | 减仓观望 |

---

## 3.5 回测配置

### 配置说明

本Section定义Phase 1和Phase 2回测的参数配置，包括时间范围、采样间隔等。

**快速测试模式**：
- 如需快速测试，可缩短时间范围（如改为3个月数据）
- 只需修改下方配置cell中的日期参数
- 完整测试使用默认配置即可

In [2]:
# ============ 环境设置 ============
import sys
from pathlib import Path

# 自动检测项目根目录（包含core目录的目录）
current_dir = Path.cwd()
project_root = None
for parent in [current_dir] + list(current_dir.parents):
    if (parent / 'core').exists() and (parent / 'config').exists():
        project_root = parent
        break

if project_root is None:
    # 回退到当前工作目录的父目录（适用于worktrees）
    project_root = Path('/home/taotao/.cursor/worktrees/TRQuant/ope')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print(f'✅ 项目路径已设置: {project_root}')

# ============ 回测配置 ============
# Phase 1 配置（快速验证）
PHASE1_START_DATE = "2023-01-01"  # 可改为短时间范围进行快速测试，如: "2024-06-01"
PHASE1_END_DATE = "2024-08-16"
PHASE1_SAMPLE_INTERVAL = 10

# Phase 2 配置（完整10年回测）
PHASE2_START_DATE = "2014-11-17"  # 可改为短时间范围进行快速测试，如: "2024-01-01"
PHASE2_END_DATE = "2024-08-16"
PHASE2_SAMPLE_INTERVAL = 30

# 快速测试模式（可选）
# 取消注释下面的行来启用快速测试（使用最近3个月数据）
# from datetime import datetime, timedelta
# PHASE1_END_DATE = datetime.now().strftime("%Y-%m-%d")
# PHASE1_START_DATE = (datetime.now() - timedelta(days=90)).strftime("%Y-%m-%d")
# PHASE2_START_DATE = PHASE1_START_DATE
# PHASE2_END_DATE = PHASE1_END_DATE

print("✅ 回测配置已加载")
print(f"\nPhase 1配置:")
print(f"  - 时间范围: {PHASE1_START_DATE} ~ {PHASE1_END_DATE}")
print(f"  - 采样间隔: 每{PHASE1_SAMPLE_INTERVAL}个交易日")
print(f"\nPhase 2配置:")
print(f"  - 时间范围: {PHASE2_START_DATE} ~ {PHASE2_END_DATE}")
print(f"  - 采样间隔: 每{PHASE2_SAMPLE_INTERVAL}个交易日")

✅ 项目路径已设置: /home/taotao/.cursor/worktrees/TRQuant/ope
✅ 回测配置已加载

Phase 1配置:
  - 时间范围: 2023-01-01 ~ 2024-08-16
  - 采样间隔: 每10个交易日

Phase 2配置:
  - 时间范围: 2014-11-17 ~ 2024-08-16
  - 采样间隔: 每30个交易日


### 3.2 A股特色指标阈值

| 指标 | 强看多 | 看多 | 中性 | 看空 | 强看空 |
|------|--------|------|------|------|--------|
| 北向5日累计(亿) | >100 | 50~100 | -50~50 | -100~-50 | <-100 |
| 融资变化率(%) | >2% | 1%~2% | -1%~1% | -2%~-1% | <-2% |
| 涨跌停比 | >3 | 2~3 | 0.5~2 | 0.3~0.5 | <0.3 |

---

## 4. Phase 1: 快速验证回测

### 4.1 回测参数

| 参数 | 值 | 说明 |
|------|-----|------|
| 时间范围 | 2023-01-01 ~ 2024-08-16 | 约1.5年 |
| 采样间隔 | 每10个交易日 | ~40个数据点 |
| 预计耗时 | 2-3分钟 | |
| 目的 | 验证框架正确性 | |

In [3]:
# Phase 1: 检查是否有已运行的回测结果
print("\n" + "="*60)
print("🔍 Phase 1: 检查回测结果")
print("="*60)

# ============ 配置区域 ============
USE_CACHE = True      # True=优先使用缓存, False=强制重新运行
# =================================

import time
start_time = time.time()

try:
    from core.signal_backtest import BacktestConfig
    from core.market_trend_storage import MarketTrendStorage
    
    # 检查MongoDB连接
    storage = MarketTrendStorage()
    db_connected = storage.is_connected()
    print(f"MongoDB: {'✅ 已连接' if db_connected else '⚠️ 未连接'}")
    
    # 使用配置变量
    config = BacktestConfig(
        start_date=PHASE1_START_DATE,
        end_date=PHASE1_END_DATE
    )
    config_dict = config.to_dict()
    config_dict['sample_interval'] = PHASE1_SAMPLE_INTERVAL
    
    print(f"\n检查参数:")
    print(f"  - 时间范围: {PHASE1_START_DATE} ~ {PHASE1_END_DATE}")
    print(f"  - 采样间隔: 每{PHASE1_SAMPLE_INTERVAL}个交易日")
    print(f"  - 使用缓存: {'是' if USE_CACHE else '否'}")
    
    # 两级检查：先检查缓存（精确匹配），再检查数据库（最新结果）
    phase1_result = None
    phase1_result_id = None
    
    if USE_CACHE and db_connected:
        # 第一级：检查缓存（基于配置哈希的精确匹配）
        cached = storage.find_cached_backtest(config_dict, backtest_type='signal_phase1')
        if cached and cached.get('summary', {}).get('total_signals', 0) > 0:
            print(f"\n💾 找到缓存（精确匹配）:")
            print(f"   ID: {cached['_id']}")
            print(f"   创建时间: {cached.get('created_at', 'N/A')[:19]}")
            print(f"   版本: {cached.get('version_tag', '未设置')}")
            print(f"   信号数: {cached.get('summary', {}).get('total_signals', 0)}")
            
            phase1_result = storage.load_backtest_result(cached['_id'])
            if phase1_result:
                phase1_result_id = str(cached['_id'])
                print(f"✅ 加载成功! (耗时: {time.time()-start_time:.1f}秒)")
        else:
            print(f"\n⚠️ 缓存未找到，尝试从数据库查找...")
        
        # 第二级：如果缓存未找到或有问题，检查数据库（查找最新结果）
        if phase1_result is None:
            results = storage.query_backtest_results(
                backtest_type='signal_phase1',
                limit=1,
                sort_by='created_at',
                sort_order=-1
            )
            if results and results[0].get('summary', {}).get('total_signals', 0) > 0:
                latest = results[0]
                print(f"\n💾 找到数据库结果（最新）:")
                print(f"   ID: {latest['_id']}")
                print(f"   创建时间: {latest.get('created_at', 'N/A')[:19]}")
                print(f"   版本: {latest.get('version_tag', '未设置')}")
                print(f"   信号数: {latest.get('summary', {}).get('total_signals', 0)}")
                
                phase1_result = storage.load_backtest_result(latest['_id'])
                if phase1_result:
                    phase1_result_id = str(latest['_id'])
                    print(f"✅ 加载成功! (耗时: {time.time()-start_time:.1f}秒)")
    
    # 显示检查结果
    if phase1_result:
        print(f"\n✅ 检查结果: 找到已运行的回测结果")
        print(f"   总信号数: {phase1_result.total_signals}")
        print(f"   短期准确率: {phase1_result.short_accuracy_5d:.1f}%")
        print(f"   中期准确率: {phase1_result.medium_accuracy_20d:.1f}%")
        print(f"   长期准确率: {phase1_result.long_accuracy_60d:.1f}%")
        print(f"\n💡 提示: 如需重新运行回测，请执行下一个cell")
    else:
        print(f"\n❌ 检查结果: 未找到已运行的回测结果")
        print(f"💡 提示: 请执行下一个cell运行Phase 1回测")
    
except Exception as e:
    print(f"❌ 检查失败: {e}")
    import traceback
    traceback.print_exc()

2026-01-05 13:11:53,311 - INFO - MongoDB连接成功: jqquant



🔍 Phase 1: 检查回测结果
MongoDB: ✅ 已连接

检查参数:
  - 时间范围: 2023-01-01 ~ 2024-08-16
  - 采样间隔: 每10个交易日
  - 使用缓存: 是

⚠️ 缓存未找到，尝试从数据库查找...

💾 找到数据库结果（最新）:
   ID: 695ae896cf1ec34386c0bc3a
   创建时间: 2026-01-04T17:24:22
   版本: None
   信号数: 40
✅ 加载成功! (耗时: 0.1秒)

✅ 检查结果: 找到已运行的回测结果
   总信号数: 40
   短期准确率: 62.5%
   中期准确率: 75.0%
   长期准确率: 62.5%

💡 提示: 如需重新运行回测，请执行下一个cell


In [4]:
# Phase 1: 执行回测（如果需要）
print("\n" + "="*60)
print("📊 Phase 1: 执行回测")
print("="*60)

# ============ 配置区域 ============
VERSION_TAG = None    # 版本标签，如: "v1.0", "优化后", None=自动时间戳
FORCE_RUN = False     # True=强制重新运行, False=如果已有结果则跳过
# =================================

import time
start_time = time.time()

try:
    from core.signal_backtest import run_phase1_backtest, BacktestConfig
    from core.market_trend_storage import MarketTrendStorage
    from datetime import datetime
    
    # 检查是否已有结果（从检查cell）
    if 'phase1_result' in globals() and phase1_result and not FORCE_RUN:
        print(f"\n✅ 已有回测结果，跳过执行")
        print(f"   总信号数: {phase1_result.total_signals}")
        print(f"   如需重新运行，请设置 FORCE_RUN = True")
    else:
        # 检查MongoDB连接
        storage = MarketTrendStorage()
        db_connected = storage.is_connected()
        print(f"MongoDB: {'✅ 已连接' if db_connected else '⚠️ 未连接'}")
        
        # 自动生成版本标签
        if VERSION_TAG is None:
            VERSION_TAG = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        # 使用配置变量
        config = BacktestConfig(
            start_date=PHASE1_START_DATE,
            end_date=PHASE1_END_DATE
        )
        config_dict = config.to_dict()
        config_dict['sample_interval'] = PHASE1_SAMPLE_INTERVAL
        
        print(f"\n回测参数:")
        print(f"  - 时间范围: {PHASE1_START_DATE} ~ {PHASE1_END_DATE}")
        print(f"  - 采样间隔: 每{PHASE1_SAMPLE_INTERVAL}个交易日")
        print(f"  - 版本标签: {VERSION_TAG}")
        
        # 执行回测
        print("\n🔄 正在执行Phase 1回测...")
        phase1_result = run_phase1_backtest(
            sample_interval=PHASE1_SAMPLE_INTERVAL,
            start_date=PHASE1_START_DATE,
            end_date=PHASE1_END_DATE
        )
        print(f"✅ 回测完成! (耗时: {time.time()-start_time:.1f}秒)")
        
        # 保存到数据库并设置版本标签
        if db_connected:
            try:
                result_id = storage.save_backtest_result(
                    result=phase1_result,
                    config=config_dict,
                    backtest_type='signal_phase1',
                    version_tag=VERSION_TAG
                )
                if result_id:
                    print(f"💾 已保存到数据库, ID: {result_id}, 版本: {VERSION_TAG}")
            except Exception as e:
                print(f"⚠️ 保存失败: {e}")
        
        print(f"\n📊 回测结果:")
        print(f"   总信号数: {phase1_result.total_signals}")
        print(f"   短期准确率: {phase1_result.short_accuracy_5d:.1f}%")
        print(f"   中期准确率: {phase1_result.medium_accuracy_20d:.1f}%")
        print(f"   长期准确率: {phase1_result.long_accuracy_60d:.1f}%")
    
except Exception as e:
    print(f"❌ Phase 1 回测失败: {e}")
    import traceback
    traceback.print_exc()


📊 Phase 1: 执行回测

✅ 已有回测结果，跳过执行
   总信号数: 40
   如需重新运行，请设置 FORCE_RUN = True


In [5]:
# Phase 1 结果报告
from core.signal_backtest import SignalBacktester
from IPython.display import display, Markdown

if 'phase1_result' in dir() and phase1_result:
    backtester = SignalBacktester()
    report = backtester.generate_report(phase1_result)
    display(Markdown(report))



# 市场趋势信号回测报告 (增强版)

## 回测概况

| 项目 | 值 |
|------|-----|
| 回测时间 | 2026-01-02 20:44:15 |
| 回测区间 | 2023-01-01 ~ 2024-08-16 |
| 基准指数 | 000001.XSHG |
| 总信号数 | 40 |
| 看多信号 | 13 |
| 看空信号 | 12 |
| 中性信号 | 15 |
| 耗时 | 90.2秒 |

## 综合准确率

| 持有期 | 准确率 |
|--------|--------|
| 5日 | 62.5% |
| 10日 | 70.0% |
| 20日 | 72.5% |
| 60日 | 62.5% |

## 分周期准确率

| 周期 | 验证期 | 准确率 | 看多准确 | 看空准确 |
|------|--------|--------|----------|----------|
| 短期 | 5日 | 62.5% | 38.9% | 100.0% |
| 中期 | 20日 | 75.0% | 42.9% | 80.0% |
| 长期 | 60日 | 62.5% | 30.8% | 62.5% |

## 市场状态准确率

| 状态类别 | 60日准确率 |
|----------|------------|
| 牛市系列 | 0.0% |
| 熊市系列 | 18.8% |
| 震荡系列 | 63.6% |
| **综合** | **25.0%** |

## 分类信号表现

### 看多信号
| 指标 | 值 |
|------|-----|
| 信号数量 | 13 |
| 5日胜率 | 30.8% |
| 5日平均收益 | -0.87% |
| 20日平均收益 | -0.79% |

### 看空信号
| 指标 | 值 |
|------|-----|
| 信号数量 | 12 |
| 5日胜率 | 83.3% |
| 5日平均收益 | -1.05% |
| 20日平均收益 | -1.76% |

## 年度统计

| 年份 | 信号数 | 5日准确 | 20日准确 | 60日准确 | 短期准确 | 中期准确 | 长期准确 |
|------|--------|---------|----------|----------|----------|----------|----------|
| 2023 | 25 | 60% | 76% | 64% | 68% | 80% | 72% |
| 2024 | 15 | 67% | 67% | 60% | 53% | 67% | 47% |


---

## 5. Phase 2: 完整10年回测

### 5.1 回测参数

| 参数 | 值 | 说明 |
|------|-----|------|
| 时间范围 | 2014-11-17 ~ 2024-08-16 | 约10年 |
| 采样间隔 | 每10个交易日 | ~240个数据点 |
| 预计耗时 | 8-12分钟 | 分3段串行 |

### 5.2 时间分割 (市场特征)

| 时间段 | 数据点 | 市场特征 |
|--------|--------|----------|
| 2014.11-2017.12 | ~250 | 牛熊转换期(2015股灾) |
| 2018.01-2021.06 | ~280 | 熊市+疫情复苏 |
| 2021.07-2024.08 | ~250 | 结构性行情 |

In [6]:
# Phase 2: 检查是否有已运行的回测结果
print("\n" + "="*60)
print("🔍 Phase 2: 检查回测结果")
print("="*60)

# ============ 配置区域 ============
USE_CACHE = True      # True=优先使用缓存, False=强制重新运行
# =================================

import time
start_time = time.time()

# 辅助函数：兼容对象和字典的属性访问
def get_attr(obj, name, default=0):
    if hasattr(obj, name):
        return getattr(obj, name)
    elif isinstance(obj, dict):
        return obj.get(name, default)
    return default

try:
    from core.signal_backtest import BacktestConfig
    from core.market_trend_storage import MarketTrendStorage
    
    storage = MarketTrendStorage()
    db_connected = storage.is_connected()
    print(f"MongoDB: {'✅ 已连接' if db_connected else '⚠️ 未连接'}")
    
    config = BacktestConfig(start_date=PHASE2_START_DATE, end_date=PHASE2_END_DATE)
    config_dict = config.to_dict()
    config_dict['sample_interval'] = PHASE2_SAMPLE_INTERVAL
    
    print(f"\n检查参数:")
    print(f"  - 时间范围: {PHASE2_START_DATE} ~ {PHASE2_END_DATE}")
    print(f"  - 采样间隔: 每{PHASE2_SAMPLE_INTERVAL}个交易日")
    print(f"  - 使用缓存: {'是' if USE_CACHE else '否'}")
    
    phase2_result = None
    phase2_result_id = None
    
    if USE_CACHE and db_connected:
        # 查找最新的Phase 2结果
        results = storage.query_backtest_results(
            backtest_type='signal_phase2',
            limit=1,
            sort_by='created_at',
            sort_order=-1
        )
        if results and results[0].get('summary', {}).get('total_signals', 0) > 0:
            latest = results[0]
            print(f"\n💾 找到数据库结果:")
            print(f"   ID: {latest['_id']}")
            print(f"   创建时间: {latest.get('created_at', 'N/A')[:19]}")
            print(f"   版本: {latest.get('version_tag', '未设置')}")
            print(f"   信号数: {latest.get('summary', {}).get('total_signals', 0)}")
            
            phase2_result = storage.load_backtest_result(latest['_id'])
            if phase2_result:
                phase2_result_id = str(latest['_id'])
                print(f"✅ 加载成功! (耗时: {time.time()-start_time:.1f}秒)")
    
    if phase2_result:
        print(f"\n✅ 检查结果: 找到已运行的回测结果")
        print(f"   总信号数: {get_attr(phase2_result, 'total_signals')}")
        print(f"   短期准确率: {get_attr(phase2_result, 'short_accuracy_5d'):.1f}%")
        print(f"   中期准确率: {get_attr(phase2_result, 'medium_accuracy_20d'):.1f}%")
        print(f"   长期准确率: {get_attr(phase2_result, 'long_accuracy_60d'):.1f}%")
        print(f"\n💡 提示: 如需重新运行回测，请执行下一个cell")
    else:
        print(f"\n❌ 检查结果: 未找到已运行的回测结果")
        print(f"💡 提示: 请执行下一个cell运行Phase 2回测")
    
except Exception as e:
    print(f"❌ 检查失败: {e}")
    import traceback
    traceback.print_exc()


2026-01-05 13:11:57,020 - INFO - MongoDB连接成功: jqquant



🔍 Phase 2: 检查回测结果
MongoDB: ✅ 已连接

检查参数:
  - 时间范围: 2014-11-17 ~ 2024-08-16
  - 采样间隔: 每30个交易日
  - 使用缓存: 是

💾 找到数据库结果:
   ID: 695b4210e3d489b8b42bf950
   创建时间: 2026-01-04T23:46:08
   版本: direct_test_20260104_234552
   信号数: 3
✅ 加载成功! (耗时: 0.0秒)

✅ 检查结果: 找到已运行的回测结果
   总信号数: 3
   短期准确率: 66.7%
   中期准确率: 100.0%
   长期准确率: 66.7%

💡 提示: 如需重新运行回测，请执行下一个cell


In [8]:
# Phase 2: 执行回测（如果需要）
# ⚠️ 注意: 此cell执行时间较长 (约8-12分钟，取决于时间范围)

RUN_PHASE2 = True  # 设为True执行完整回测

# ============ 配置区域 ============
VERSION_TAG = None    # 版本标签，如: "v1.0", "优化后", None=自动时间戳
FORCE_RUN = True     # True=强制重新运行, False=如果已有结果则跳过
# =================================

# 辅助函数：兼容对象和字典的属性访问
def get_attr(obj, name, default=0):
    if hasattr(obj, name):
        return getattr(obj, name)
    elif isinstance(obj, dict):
        return obj.get(name, default)
    return default

if not RUN_PHASE2:
    print("⚠️ Phase 2 未启用")
    print("如需执行完整10年回测，请将 RUN_PHASE2 改为 True")
else:
    print("\n" + "="*60)
    print("📊 Phase 2: 执行回测")
    print("="*60)
    
    import time
    start_time = time.time()
    
    try:
        
        from core.signal_backtest import run_phase2_backtest, BacktestConfig
        from core.market_trend_storage import MarketTrendStorage
        from datetime import datetime
        
        # 检查是否已有结果（从检查cell）
        if 'phase2_result' in globals() and phase2_result and not FORCE_RUN:
            print(f"\n✅ 已有回测结果，跳过执行")
            print(f"   总信号数: {get_attr(phase2_result, 'total_signals')}")
            print(f"   如需重新运行，请设置 FORCE_RUN = True")
        else:
            # 检查MongoDB连接
            storage = MarketTrendStorage()
            db_connected = storage.is_connected()
            print(f"MongoDB: {'✅ 已连接' if db_connected else '⚠️ 未连接'}")
            
            # 自动生成版本标签
            if VERSION_TAG is None:
                VERSION_TAG = f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            
            # 使用配置变量
            config = BacktestConfig(
                start_date=PHASE2_START_DATE,
                end_date=PHASE2_END_DATE
            )
            config_dict = config.to_dict()
            config_dict['sample_interval'] = PHASE2_SAMPLE_INTERVAL
            
            print(f"\n回测参数:")
            print(f"  - 时间范围: {PHASE2_START_DATE} ~ {PHASE2_END_DATE}")
            print(f"  - 采样间隔: 每{PHASE2_SAMPLE_INTERVAL}个交易日")
            print(f"  - 分3个时间段串行处理")
            print(f"  - 版本标签: {VERSION_TAG}")
            
            # 执行回测
            print("\n🔄 正在执行Phase 2完整回测...")
            print("预计耗时: 8-12分钟\n")
            
            phase2_result = run_phase2_backtest(
                sample_interval=PHASE2_SAMPLE_INTERVAL,
                start_date=PHASE2_START_DATE,
                end_date=PHASE2_END_DATE
            )
            
            elapsed = time.time() - start_time
            print(f"✅ 回测完成! (耗时: {elapsed/60:.1f}分钟)")
            
            # 保存到数据库并设置版本标签
            if db_connected:
                try:
                    result_id = storage.save_backtest_result(
                        result=phase2_result,
                        config=config_dict,
                        backtest_type='signal_phase2',
                        version_tag=VERSION_TAG
                    )
                    if result_id:
                        print(f"💾 已保存到数据库, ID: {result_id}, 版本: {VERSION_TAG}")
                except Exception as e:
                    print(f"⚠️ 保存失败: {e}")
            
            print(f"\n📊 回测结果:")
            print(f"   总信号数: {phase2_result.total_signals}")
            print(f"   短期准确率: {phase2_result.short_accuracy_5d:.1f}%")
            print(f"   中期准确率: {phase2_result.medium_accuracy_20d:.1f}%")
            print(f"   长期准确率: {phase2_result.long_accuracy_60d:.1f}%")
            print(f"   市场状态准确率: {phase2_result.state_accuracy_60d:.1f}%")
        
    except Exception as e:
        print(f"❌ Phase 2 回测失败: {e}")
        import traceback
        traceback.print_exc()


2026-01-05 13:15:31,667 - INFO - MongoDB连接成功: jqquant
2026-01-05 13:15:31,668 - INFO - 开始Phase 2完整回测 (使用3个进程)
2026-01-05 13:15:31,668 - INFO - 时间范围: 2014-11-17 ~ 2024-08-16
2026-01-05 13:15:31,668 - INFO - Worker 1: 回测 2014-11-17 ~ 2018-02-15
2026-01-05 13:15:31,669 - INFO - JQData认证成功
2026-01-05 13:15:31,669 - INFO - TrendAnalyzer (8维指标) 初始化成功
2026-01-05 13:15:31,669 - INFO - SimpleHMM 初始化成功
2026-01-05 13:15:31,669 - INFO - IBDStyleAnalyzer 初始化成功
2026-01-05 13:15:31,669 - INFO - 开始回测: 2014-11-17 ~ 2018-02-15
2026-01-05 13:15:31,671 - INFO - 共 797 个交易日



📊 Phase 2: 执行回测
MongoDB: ✅ 已连接

回测参数:
  - 时间范围: 2014-11-17 ~ 2024-08-16
  - 采样间隔: 每30个交易日
  - 分3个时间段串行处理
  - 版本标签: run_20260105_131531

🔄 正在执行Phase 2完整回测...
预计耗时: 8-12分钟



2026-01-05 13:15:32,975 - INFO - 采样 27 个交易日进行回测
2026-01-05 13:15:32,976 - INFO - 回测进度: 1/27 (2014-11-17)
2026-01-05 13:15:34,466 - INFO - 市场趋势分析完成: 000001.XSHG, 综合得分=38.8, 阶段=牛市确认(全周期共振)
2026-01-05 13:15:34,703 - INFO - 🔍 开始IBD风格市场分析: 000001.XSHG
2026-01-05 13:15:34,935 - INFO - 获取价格数据成功: 000001.XSHG, 2025-04-20 to 2026-01-05
2026-01-05 13:15:34,938 - INFO - 🔍 IBD分析完成: rally_attempt, 得分: 10.0
2026-01-05 13:15:36,244 - INFO - 市场趋势分析完成: 000001.XSHG, 综合得分=36.0, 阶段=牛市确认(全周期共振)
2026-01-05 13:15:36,479 - INFO - 🔍 开始IBD风格市场分析: 000001.XSHG
2026-01-05 13:15:36,710 - INFO - 获取价格数据成功: 000001.XSHG, 2025-04-20 to 2026-01-05
2026-01-05 13:15:36,714 - INFO - 🔍 IBD分析完成: rally_attempt, 得分: 10.0
2026-01-05 13:15:38,007 - INFO - 市场趋势分析完成: 000001.XSHG, 综合得分=-20.9, 阶段=破位风险
2026-01-05 13:15:38,246 - INFO - 🔍 开始IBD风格市场分析: 000001.XSHG
2026-01-05 13:15:38,478 - INFO - 获取价格数据成功: 000001.XSHG, 2025-04-20 to 2026-01-05
2026-01-05 13:15:38,481 - INFO - 🔍 IBD分析完成: rally_attempt, 得分: 10.0
2026-01-05 13:15:39,770 - IN

✅ 回测完成! (耗时: 3.0分钟)
💾 已保存到数据库, ID: 695bffa019118d65030df6ac, 版本: run_20260105_131531

📊 回测结果:
   总信号数: 81
   短期准确率: 54.3%
   中期准确率: 54.3%
   长期准确率: 67.9%
   市场状态准确率: 46.9%


In [ ]:
# Phase 2 结果报告
from core.signal_backtest import SignalBacktester
from IPython.display import display, Markdown

if 'phase2_result' in dir() and phase2_result:
    backtester = SignalBacktester()
    report = backtester.generate_report(phase2_result)
    display(Markdown(report))


---

## 6. 结果分析与可视化

In [ ]:
# 可视化: 准确率热力图（支持从数据库自动加载）
try:
    from core.backtest_visualization import BacktestVisualization
    from core.market_trend_storage import MarketTrendStorage
    
    # 如果内存中没有phase2_result，从数据库加载最新的
    if 'phase2_result' not in dir() or not phase2_result:
        storage = MarketTrendStorage()
        if storage.is_connected():
            results = storage.query_backtest_results(
                backtest_type='signal_phase2',
                limit=1,
                sort_by='created_at',
                sort_order=-1
            )
            if results:
                phase2_result = storage.load_backtest_result(str(results[0]['_id']))
                if phase2_result:
                    version_tag = results[0].get('version_tag', '未设置')
                    print(f"✅ 从数据库加载最新Phase 2结果 (版本: {version_tag})")
            else:
                print("⚠️ 数据库中无Phase 2结果，请先运行Phase 2回测")
    
    # 创建可视化对象
    if 'phase2_result' in dir() and phase2_result:
        viz = BacktestVisualization(phase2_result)
        
        # 准确率热力图
        fig = viz.create_accuracy_heatmap()
        if fig:
            fig.show()
    else:
        print("⚠️ 未找到Phase 2结果，无法进行可视化")
        
except Exception as e:
    print(f"可视化失败: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# 可视化: 年度准确率（支持从数据库自动加载）
try:
    from core.backtest_visualization import BacktestVisualization
    from core.market_trend_storage import MarketTrendStorage
    
    # 如果viz未定义，需要重新创建
    if 'viz' not in dir():
        # 如果内存中没有phase2_result，从数据库加载最新的
        if 'phase2_result' not in dir() or not phase2_result:
            storage = MarketTrendStorage()
            if storage.is_connected():
                results = storage.query_backtest_results(
                    backtest_type='signal_phase2',
                    limit=1,
                    sort_by='created_at',
                    sort_order=-1
                )
                if results:
                    phase2_result = storage.load_backtest_result(str(results[0]['_id']))
                    if phase2_result:
                        version_tag = results[0].get('version_tag', '未设置')
                        print(f"✅ 从数据库加载最新Phase 2结果 (版本: {version_tag})")
        
        # 创建viz对象
        if 'phase2_result' in dir() and phase2_result:
            viz = BacktestVisualization(phase2_result)
    
    # 绘制年度准确率图表
    if 'viz' in dir() and viz:
        fig = viz.create_yearly_accuracy_bar()
        if fig:
            fig.show()
    else:
        print("⚠️ 未找到可视化对象，请先运行准确率热力图cell")
        
except Exception as e:
    print(f"年度可视化失败: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# 可视化: 市场状态时间线（支持从数据库自动加载）
try:
    from core.backtest_visualization import BacktestVisualization
    from core.market_trend_storage import MarketTrendStorage
    
    # 如果viz未定义，需要重新创建
    if 'viz' not in dir():
        # 如果内存中没有phase2_result，从数据库加载最新的
        if 'phase2_result' not in dir() or not phase2_result:
            storage = MarketTrendStorage()
            if storage.is_connected():
                results = storage.query_backtest_results(
                    backtest_type='signal_phase2',
                    limit=1,
                    sort_by='created_at',
                    sort_order=-1
                )
                if results:
                    phase2_result = storage.load_backtest_result(str(results[0]['_id']))
                    if phase2_result:
                        version_tag = results[0].get('version_tag', '未设置')
                        print(f"✅ 从数据库加载最新Phase 2结果 (版本: {version_tag})")
        
        # 创建viz对象
        if 'phase2_result' in dir() and phase2_result:
            viz = BacktestVisualization(phase2_result)
    
    # 绘制市场状态时间线图表
    if 'viz' in dir() and viz:
        fig = viz.create_market_state_timeline()
        if fig:
            fig.show()
    else:
        print("⚠️ 未找到可视化对象，请先运行准确率热力图cell")
        
except Exception as e:
    print(f"时间线可视化失败: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# 保存HTML报告
if 'phase2_result' in dir() and phase2_result:
    try:
        html_path = "/home/taotao/dev/QuantTest/TRQuant/output/market_trend_backtest_report.html"
        viz.generate_html_report(html_path)
        print(f"\n📄 HTML报告已保存: {html_path}")
    except Exception as e:
        print(f"HTML报告生成失败: {e}")

---

## 7. 参数优化建议

In [ ]:
# 参数优化分析
if 'phase2_result' in dir() and phase2_result:
    
    print("\n" + "="*60)
    print("📈 参数优化建议")
    print("="*60)
    
    # 分析各周期表现
    print("\n1. 周期权重调整建议:")
    print(f"   短期准确率: {phase2_result.short_accuracy_5d:.1f}% (当前权重20%)")
    print(f"   中期准确率: {phase2_result.medium_accuracy_20d:.1f}% (当前权重30%)")
    print(f"   长期准确率: {phase2_result.long_accuracy_60d:.1f}% (当前权重50%)")
    
    if phase2_result.long_accuracy_60d > phase2_result.short_accuracy_5d + 10:
        print("   ✅ 长期信号更可靠，可考虑增加长期权重到55-60%")
    
    # 分析看多看空表现
    print("\n2. 信号类型分析:")
    print(f"   看多5日胜率: {phase2_result.win_rate_bullish:.1f}%")
    print(f"   看空5日胜率: {phase2_result.win_rate_bearish:.1f}%")
    
    if phase2_result.win_rate_bearish < 50:
        print("   ⚠️ 看空信号胜率较低，建议:")
        print("      - 提高看空阈值 (如从-30提高到-40)")
        print("      - 增加确认条件 (多周期共振)")
    
    # 分析市场状态
    print("\n3. 市场状态识别优化:")
    print(f"   牛市识别准确率: {phase2_result.bull_state_accuracy:.1f}%")
    print(f"   熊市识别准确率: {phase2_result.bear_state_accuracy:.1f}%")
    print(f"   震荡识别准确率: {phase2_result.volatile_state_accuracy:.1f}%")

---

## 8. 结论与后续改进

In [ ]:
# 总结报告
if 'phase2_result' in dir() and phase2_result:
    
    summary = f"""
## 📋 回测总结报告

### 回测概况
- **回测区间**: 2014-11-17 ~ 2024-08-16 (约10年)
- **总信号数**: {phase2_result.total_signals}
- **看多/看空/中性**: {phase2_result.bullish_signals}/{phase2_result.bearish_signals}/{phase2_result.neutral_signals}

### 准确率汇总

| 验证周期 | 准确率 | 评价 |
|----------|--------|------|
| 短期(5日) | {phase2_result.short_accuracy_5d:.1f}% | {'✅ 及格' if phase2_result.short_accuracy_5d > 50 else '⚠️ 需优化'} |
| 中期(20日) | {phase2_result.medium_accuracy_20d:.1f}% | {'✅ 良好' if phase2_result.medium_accuracy_20d > 55 else '⚠️ 需优化'} |
| 长期(60日) | {phase2_result.long_accuracy_60d:.1f}% | {'✅ 良好' if phase2_result.long_accuracy_60d > 60 else '⚠️ 需优化'} |

### 关键发现

1. **长期信号更可靠**: 长期准确率({phase2_result.long_accuracy_60d:.0f}%)高于短期({phase2_result.short_accuracy_5d:.0f}%)
2. **震荡市识别最好**: {phase2_result.volatile_state_accuracy:.0f}%准确率
3. **看空信号需优化**: 看空5日胜率仅{phase2_result.win_rate_bearish:.0f}%

### 后续改进方向

1. 调整看空信号阈值，增加确认条件
2. 考虑增加长期信号权重
3. 针对不同市场周期使用动态参数
4. 增加更多A股特色指标（如主力资金、板块轮动）
"""
    
    display(Markdown(summary))

---

## 📚 参考资料

### 核心模块

| 模块 | 路径 | 说明 |
|------|------|------|
| 回测框架 | `core/signal_backtest.py` | SignalBacktester, run_phase1_backtest, run_phase2_backtest |
| 可视化 | `core/backtest_visualization.py` | BacktestVisualization |
| 市场状态定义 | `core/market_state_definitions.py` | 14种状态量化定义 |
| A股指标 | `core/astock_indicators.py` | 北向资金、融资融券、市场宽度 |

### 相关Notebook

| Notebook | 说明 |
|----------|------|
| `01_market_trend_comprehensive.ipynb` | 应用市场环境判断 (使用本notebook验证的方法) |
| `00_system_architecture_workflow.ipynb` | 系统架构总览 |